In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [14]:
data_dir = Path("/Users/jingying/Documents/IDX Exchange/py/cleaned_data/run_20261014_180636_999980")

# Load the already-preprocessed feature matrices.
X_train = sparse.load_npz(data_dir / "X_train.npz")
X_test = sparse.load_npz(data_dir / "X_test.npz")

# Load the corresponding record IDs and actual prices.
train_targets = pd.read_csv(data_dir / "y_train.csv")
test_targets = pd.read_csv(data_dir / "y_test.csv")

# Select ClosePrice as the prediction target.
y_train = train_targets["ClosePrice"].to_numpy()
y_test = test_targets["ClosePrice"].to_numpy()

# Check that feature rows and target values have matching lengths.
assert X_train.shape[0] == len(y_train)
assert X_test.shape[0] == len(y_test)

print("Training data:", X_train.shape)
print("Test data:", X_test.shape)

Training data: (264531, 497)
Test data: (12458, 497)


In [16]:
# Read the IDs of test sales within the frozen training price bounds.
range_targets = pd.read_csv(data_dir / "y_test_avm_range.csv")

# Identify those records in the full test set without changing row order.
in_test_range = test_targets["row_id"].isin(range_targets["row_id"]).to_numpy()

In [18]:
# Use the same training and test data for all three models.
models = {
    "Linear Regression": LinearRegression(),

    # Limit tree complexity to reduce overfitting.
    "Decision Tree": DecisionTreeRegressor(
        max_depth=15,
        min_samples_leaf=10,
        random_state=42
    ),

    # Average 100 trees to produce more stable predictions.
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        min_samples_leaf=5,
        max_features=0.7,
        random_state=42,
        n_jobs=2
    )
}

In [20]:
results = []
predictions = {}

for name, model in models.items():

    # Fit the model using training data only.
    model.fit(X_train, y_train)

    # Predict training prices to help identify overfitting.
    train_pred = model.predict(X_train)

    # Predict prices for the same held-out test month.
    test_pred = model.predict(X_test)

    # Keep predictions for later error analysis.
    predictions[name] = test_pred

    # Calculate absolute percentage errors.
    ape = 100 * np.abs((y_test - test_pred) / y_test)

    # Record model performance.
    results.append({
        "Model": name,
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred),
        "MAPE (%)": np.mean(ape),
        "MdAPE (%)": np.median(ape),
        "MAE ($)": mean_absolute_error(y_test, test_pred)
    })

    print(name, "finished")

comparison = pd.DataFrame(results)
display(comparison.round(3))


Linear Regression finished
Decision Tree finished
Random Forest finished


,Model,Train R2,Test R2,MAPE (%),MdAPE (%),MAE ($)
0,Linear Regression,0.787,0.009,25.036,17.542,490192.755
1,Decision Tree,0.901,0.008,15.959,10.404,429967.133
2,Random Forest,0.939,0.009,13.648,8.459,398892.372


In [22]:
# Report how much of the full test month is included in this subset.
print("Full test rows:", len(y_test))
print("Test rows within training price bounds:", int(np.sum(in_test_range)))
print("Coverage (%):", round(100 * np.mean(in_test_range), 2))

range_results = []

for name, test_pred in predictions.items():

    # Use the same training-derived price bounds for every model.
    actual = np.asarray(y_test)[in_test_range]
    predicted = test_pred[in_test_range]

    # Skip metric calculation if the subset is empty.
    if len(actual) == 0:
        continue

    ape = 100 * np.abs((actual - predicted) / actual)

    range_results.append({
        "Model": name,
        "Test rows": len(actual),
        "Test R2": r2_score(actual, predicted) if len(actual) > 1 else np.nan,
        "MAPE (%)": np.mean(ape),
        "MdAPE (%)": np.median(ape),
        "MAE ($)": mean_absolute_error(actual, predicted)
    })

display(pd.DataFrame(range_results).round(3))


Full test rows: 12458
Test rows within training price bounds: 12321
Coverage (%): 98.9


,Model,Test rows,Test R2,MAPE (%),MdAPE (%),MAE ($)
0,Linear Regression,12321,0.803,24.270,17.372,260169.434
1,Decision Tree,12321,0.839,15.090,10.288,200406.354
2,Random Forest,12321,0.881,12.785,8.329,170242.430
